# Data Quality & Monitoring Framework
A simple, practical guide to validating data, logging results, checking SLAs, and generating reports.

**Sections**
1. Sample Data (with intentional problems)
2. Validation Rules (9 types)
3. Logging
4. SLA Checks
5. Report & Alerts

## Setup

In [ ]:
import json
import logging
from datetime import datetime, timedelta
from pathlib  import Path
import pandas as pd

Path("output/logs").mkdir(parents=True, exist_ok=True)
Path("output/reports").mkdir(parents=True, exist_ok=True)
print("Folders ready.")

## Section 1 — Sample Data
Three small tables with **intentional problems** so the validation rules have real issues to catch.

| Table | Problems seeded in |
|---|---|
| `customers` | missing email, duplicate ID, negative age, future date, bad email format, invalid country |
| `orders` | missing product, negative amount, zero qty, invalid status, orphan customer ID |
| `transactions` | duplicate row, null payment, invalid payment method, amount too large |

In [ ]:
# customers — 10 rows, 6 problems
customers = pd.DataFrame([
    {"id": 1, "name": "Alice",  "email": "alice@mail.com", "age": 29,  "country": "US", "joined": "2023-01-15"},
    {"id": 2, "name": "Bob",    "email": "bob@mail.com",   "age": 35,  "country": "UK", "joined": "2023-03-22"},
    {"id": 3, "name": "Carol",  "email": "",               "age": 28,  "country": "US", "joined": "2023-05-10"},  # missing email
    {"id": 4, "name": "David",  "email": "david@mail.com", "age": -5,  "country": "CA", "joined": "2023-07-01"},  # negative age
    {"id": 5, "name": "Eva",    "email": "eva@mail.com",   "age": 31,  "country": "US", "joined": "2099-12-31"},  # future date
    {"id": 2, "name": "Bob",    "email": "bob@mail.com",   "age": 35,  "country": "UK", "joined": "2023-03-22"},  # duplicate id
    {"id": 6, "name": "Frank",  "email": "not-an-email",   "age": 45,  "country": "AU", "joined": "2024-01-05"},  # bad email
    {"id": 7, "name": "Grace",  "email": "grace@mail.com", "age": None,"country": "US", "joined": "2024-02-14"},  # null age
    {"id": 8, "name": "Henry",  "email": "henry@mail.com", "age": 52,  "country": "XX", "joined": "2024-03-30"},  # invalid country
    {"id": 9, "name": "Ivy",    "email": "ivy@mail.com",   "age": 27,  "country": "US", "joined": "2024-04-01"},  # all good
])
customers

In [ ]:
# orders — 10 rows, 5 problems
orders = pd.DataFrame([
    {"order_id": 101, "customer_id": 1,  "product": "Laptop",   "amount": 999.99, "qty": 1, "status": "completed"},
    {"order_id": 102, "customer_id": 2,  "product": "Mouse",    "amount":  25.00, "qty": 2, "status": "shipped"},
    {"order_id": 103, "customer_id": 3,  "product": "Monitor",  "amount": 349.99, "qty": 1, "status": "pending"},
    {"order_id": 104, "customer_id": 1,  "product": "Keyboard", "amount":  79.99, "qty": 1, "status": "completed"},
    {"order_id": 105, "customer_id": 99, "product": "Webcam",   "amount":  89.99, "qty": 1, "status": "shipped"},   # orphan customer
    {"order_id": 106, "customer_id": 4,  "product": "",         "amount": 199.99, "qty": 1, "status": "completed"}, # missing product
    {"order_id": 107, "customer_id": 5,  "product": "Tablet",   "amount": -49.99, "qty": 1, "status": "pending"},   # negative amount
    {"order_id": 108, "customer_id": 6,  "product": "Monitor",  "amount": 349.99, "qty": 0, "status": "shipped"},   # zero qty
    {"order_id": 109, "customer_id": 7,  "product": "Headphone","amount":  89.99, "qty": 1, "status": "INVALID"},   # bad status
    {"order_id": 110, "customer_id": 8,  "product": "Speaker",  "amount": 129.99, "qty": 1, "status": "completed"},
])
orders

In [ ]:
# transactions — 9 rows, 4 problems
transactions = pd.DataFrame([
    {"txn_id": 1001, "order_id": 101, "payment": "credit_card", "amount": 999.99},
    {"txn_id": 1002, "order_id": 102, "payment": "paypal",      "amount":  25.00},
    {"txn_id": 1003, "order_id": 103, "payment": "credit_card", "amount": 349.99},
    {"txn_id": 1004, "order_id": 104, "payment": "debit_card",  "amount":  79.99},
    {"txn_id": 1004, "order_id": 104, "payment": "debit_card",  "amount":  79.99},  # exact duplicate
    {"txn_id": 1005, "order_id": 105, "payment": "crypto",      "amount":  89.99},  # invalid payment
    {"txn_id": 1006, "order_id": 106, "payment": None,          "amount": 199.99},  # null payment
    {"txn_id": 1007, "order_id": 107, "payment": "credit_card", "amount": 999999},  # amount too large
    {"txn_id": 1008, "order_id": 108, "payment": "paypal",      "amount":  89.99},
])
transactions

## Section 2 — Validation Rules
Each rule is one small function.  
Every function returns the same dict so results are easy to compare:
```python
{"rule": "...", "passed": True/False, "fails": 0, "detail": "OK"}
```

In [ ]:
# Rule 1 — No nulls or blank strings
def check_not_null(df, col):
    bad = df[col].isnull() | (df[col].astype(str).str.strip() == "")
    n   = bad.sum()
    return {"rule": f"not_null({col})", "passed": n == 0, "fails": int(n),
            "detail": f"{n} null/empty values" if n else "OK"}

# Rule 2 — No duplicate values in a column
def check_unique(df, col):
    dupes    = df[col].duplicated(keep=False)
    n        = dupes.sum()
    dup_vals = df.loc[dupes, col].unique().tolist()
    return {"rule": f"unique({col})", "passed": n == 0, "fails": int(n),
            "detail": f"Duplicates: {dup_vals}" if n else "OK"}

# Rule 3 — Values within a numeric range
def check_range(df, col, min_val=None, max_val=None):
    nums = pd.to_numeric(df[col], errors="coerce")
    bad  = pd.Series(False, index=df.index)
    if min_val is not None: bad |= nums < min_val
    if max_val is not None: bad |= nums > max_val
    bad |= nums.isnull()
    n    = bad.sum()
    return {"rule": f"range({col}, {min_val}–{max_val})", "passed": n == 0, "fails": int(n),
            "detail": f"{n} values outside [{min_val}, {max_val}]" if n else "OK"}

# Rule 4 — String matches a regex pattern
def check_format(df, col, pattern, label=""):
    vals   = df[col].dropna().astype(str)
    vals   = vals[vals.str.strip() != ""]
    bad    = ~vals.str.match(pattern)
    n      = bad.sum()
    bad_ex = vals[bad].tolist()[:3]
    return {"rule": f"format({col}, {label or pattern})", "passed": n == 0, "fails": int(n),
            "detail": f"Bad values: {bad_ex}" if n else "OK"}

# Rule 5 — Only allowed categorical values
def check_values(df, col, allowed):
    bad    = ~df[col].isin(allowed)
    n      = bad.sum()
    bad_ex = df.loc[bad, col].unique().tolist()
    return {"rule": f"values({col})", "passed": n == 0, "fails": int(n),
            "detail": f"Invalid: {bad_ex}" if n else "OK"}

# Rule 6 — Foreign key must exist in parent table
def check_fk(df, col, parent_df, parent_col):
    valid  = set(parent_df[parent_col].dropna())
    bad    = ~df[col].isin(valid)
    n      = bad.sum()
    bad_ex = df.loc[bad, col].unique().tolist()
    return {"rule": f"fk({col} → {parent_col})", "passed": n == 0, "fails": int(n),
            "detail": f"Missing in parent: {bad_ex}" if n else "OK"}

# Rule 7 — No future dates
def check_no_future(df, col):
    today = datetime.now().date()
    dates = pd.to_datetime(df[col], errors="coerce").dt.date
    bad   = dates > today
    n     = bad.sum()
    return {"rule": f"no_future({col})", "passed": n == 0, "fails": int(n),
            "detail": f"{n} future dates" if n else "OK"}

# Rule 8 — No exact duplicate rows
def check_no_duplicates(df):
    dupes = df.duplicated(keep=False)
    n     = dupes.sum()
    return {"rule": "no_duplicate_rows", "passed": n == 0, "fails": int(n),
            "detail": f"{n} duplicate rows" if n else "OK"}

# Rule 9 — Row count within expected range
def check_row_count(df, min_rows=1, max_rows=None):
    n      = len(df)
    passed = n >= min_rows and (max_rows is None or n <= max_rows)
    return {"rule": f"row_count({min_rows}–{max_rows or '∞'})", "passed": passed,
            "fails": 0 if passed else 1,
            "detail": f"Got {n} rows, expected {min_rows}–{max_rows}" if not passed else "OK"}

print("All 9 rule functions defined.")

### Run all rules

In [ ]:
EMAIL           = r"^[\w\.-]+@[\w\.-]+\.\w{2,}$"
VALID_COUNTRIES = ["US", "UK", "CA", "AU", "DE", "FR", "JP", "IN"]
VALID_STATUSES  = ["completed", "shipped", "pending", "cancelled"]
VALID_PAYMENTS  = ["credit_card", "debit_card", "paypal", "bank_transfer"]

results = []

def run(table_name, df, rules):
    for r in rules:
        r["table"] = table_name
        results.append(r)

run("customers", customers, [
    check_not_null (customers, "id"),
    check_not_null (customers, "name"),
    check_not_null (customers, "email"),
    check_unique   (customers, "id"),
    check_range    (customers, "age", min_val=0, max_val=120),
    check_format   (customers, "email", EMAIL, "email"),
    check_no_future(customers, "joined"),
    check_values   (customers, "country", VALID_COUNTRIES),
    check_row_count(customers, min_rows=1),
])

run("orders", orders, [
    check_not_null(orders, "order_id"),
    check_not_null(orders, "product"),
    check_unique  (orders, "order_id"),
    check_range   (orders, "amount", min_val=0.01, max_val=50_000),
    check_range   (orders, "qty",    min_val=1,    max_val=9_999),
    check_values  (orders, "status", VALID_STATUSES),
    check_fk      (orders, "customer_id", customers, "id"),
])

run("transactions", transactions, [
    check_not_null    (transactions, "txn_id"),
    check_not_null    (transactions, "payment"),
    check_unique      (transactions, "txn_id"),
    check_no_duplicates(transactions),
    check_range       (transactions, "amount", min_val=0.01, max_val=100_000),
    check_values      (transactions, "payment", VALID_PAYMENTS),
    check_fk          (transactions, "order_id", orders, "order_id"),
])

print(f"{len(results)} rules executed.")

### View results as a table

In [ ]:
df_results = pd.DataFrame(results)[["table", "rule", "passed", "fails", "detail"]]
df_results.style.apply(
    lambda row: ["background: #d4edda" if row["passed"] else "background: #f8d7da"] * len(row),
    axis=1
)

In [ ]:
# Summary by table
summary = df_results.groupby("table").agg(
    total   = ("passed", "count"),
    passed  = ("passed", "sum"),
    failed  = ("passed", lambda x: (~x).sum()),
).assign(pass_rate=lambda d: (d["passed"] / d["total"] * 100).round(1).astype(str) + "%")

summary

In [ ]:
# Show only failures
failures = df_results[~df_results["passed"]][["table", "rule", "fails", "detail"]]
print(f"Total failures: {len(failures)}")
failures

## Section 3 — Logging
Two log files:
- `output/logs/pipeline.log` — every rule result (INFO + WARNING)
- `output/logs/alerts.log` — failures only (WARNING)

In [ ]:
# pipeline logger — writes everything
pipeline_log = logging.getLogger("pipeline")
pipeline_log.setLevel(logging.DEBUG)
pipeline_log.handlers.clear()
fh = logging.FileHandler("output/logs/pipeline.log")
fh.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-8s  %(message)s",
                                   datefmt="%Y-%m-%d %H:%M:%S"))
pipeline_log.addHandler(fh)

# alerts logger — writes failures only
alert_log = logging.getLogger("alerts")
alert_log.setLevel(logging.WARNING)
alert_log.handlers.clear()
fh2 = logging.FileHandler("output/logs/alerts.log")
fh2.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-8s  %(message)s",
                                    datefmt="%Y-%m-%d %H:%M:%S"))
alert_log.addHandler(fh2)

# Log every result
for r in results:
    msg = f"table={r['table']}  rule={r['rule']}  fails={r['fails']}  detail={r['detail']}"
    if r["passed"]:
        pipeline_log.info(msg)
    else:
        pipeline_log.warning(msg)
        alert_log.warning(msg)

rules_passed = sum(1 for r in results if r["passed"])
pipeline_log.info(f"DONE: {rules_passed}/{len(results)} rules passed")
print("Logs written.")

In [ ]:
# Preview alerts.log
alert_lines = Path("output/logs/alerts.log").read_text().strip().splitlines()
print(f"alerts.log — {len(alert_lines)} failure lines:\n")
for line in alert_lines:
    print(" ", line)

## Section 4 — SLA Checks
An SLA defines what "good enough" means for a pipeline.  
We check 4 things per table:

| Check | Question |
|---|---|
| **Freshness** | Is the data newer than X hours? |
| **Pass rate** | Did at least Y% of quality rules pass? |
| **Volume** | Is the row count within the expected range? |
| **Latency** | Did the pipeline finish within Z seconds? |

In [ ]:
# SLA definition — one dict per table
SLA = {
    "customers":    {"max_age_hours": 24,  "min_pass_rate": 90, "min_rows": 1, "max_rows": 1_000_000, "max_runtime_sec": 300},
    "orders":       {"max_age_hours": 6,   "min_pass_rate": 95, "min_rows": 1, "max_rows": 5_000_000, "max_runtime_sec": 600},
    "transactions": {"max_age_hours": 1,   "min_pass_rate": 98, "min_rows": 1, "max_rows": 50_000_000,"max_runtime_sec": 120},
}

# Simulated pipeline metadata (in production these come from your scheduler)
metadata = {
    "customers":    {"last_loaded": datetime.now() - timedelta(hours=2),  "runtime_sec": 45},
    "orders":       {"last_loaded": datetime.now() - timedelta(hours=8),  "runtime_sec": 720},
    "transactions": {"last_loaded": datetime.now() - timedelta(hours=2),  "runtime_sec": 90},
}

tables = {"customers": customers, "orders": orders, "transactions": transactions}

sla_results = []

for table, df in tables.items():
    sla  = SLA[table]
    meta = metadata[table]

    # Freshness
    age_hours = (datetime.now() - meta["last_loaded"]).total_seconds() / 3600
    passed    = age_hours <= sla["max_age_hours"]
    sla_results.append({"table": table, "check": "freshness", "passed": passed,
                         "value": f"{age_hours:.1f}h", "limit": f"{sla['max_age_hours']}h",
                         "detail": f"Data is {age_hours:.1f}h old (limit: {sla['max_age_hours']}h)"})

    # Pass rate
    t_results = [r for r in results if r["table"] == table]
    t_passed  = sum(1 for r in t_results if r["passed"])
    rate      = round(t_passed / len(t_results) * 100) if t_results else 0
    passed    = rate >= sla["min_pass_rate"]
    sla_results.append({"table": table, "check": "pass_rate", "passed": passed,
                         "value": f"{rate}%", "limit": f"{sla['min_pass_rate']}%",
                         "detail": f"Pass rate {rate}% (minimum: {sla['min_pass_rate']}%)"})

    # Volume
    n      = len(df)
    passed = sla["min_rows"] <= n <= sla["max_rows"]
    sla_results.append({"table": table, "check": "volume", "passed": passed,
                         "value": f"{n} rows", "limit": f"{sla['min_rows']}–{sla['max_rows']}",
                         "detail": f"{n} rows (expected {sla['min_rows']}–{sla['max_rows']})"})

    # Latency
    secs   = meta["runtime_sec"]
    passed = secs <= sla["max_runtime_sec"]
    sla_results.append({"table": table, "check": "latency", "passed": passed,
                         "value": f"{secs}s", "limit": f"{sla['max_runtime_sec']}s",
                         "detail": f"Ran in {secs}s (limit: {sla['max_runtime_sec']}s)"})

print(f"SLA checks complete.")

In [ ]:
# View SLA results
df_sla = pd.DataFrame(sla_results)[["table", "check", "value", "limit", "passed", "detail"]]
df_sla.style.apply(
    lambda row: ["background: #d4edda" if row["passed"] else "background: #f8d7da"] * len(row),
    axis=1
)

In [ ]:
sla_passed = sum(1 for s in sla_results if s["passed"])
print(f"SLA summary: {sla_passed}/{len(sla_results)} checks passed")

breaches = [s for s in sla_results if not s["passed"]]
print(f"\nBreaches ({len(breaches)}):")
for b in breaches:
    print(f"  [{b['table']}] {b['check']}: {b['detail']}")

## Section 5 — Report & Alerts

In [ ]:
# Save full report as JSON
rules_passed = sum(1 for r in results if r["passed"])
sla_passed   = sum(1 for s in sla_results if s["passed"])

report = {
    "run_at":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "summary": {
        "rules_total":  len(results),
        "rules_passed": rules_passed,
        "rules_failed": len(results) - rules_passed,
        "pass_rate":    f"{round(rules_passed / len(results) * 100)}%",
        "sla_total":    len(sla_results),
        "sla_passed":   sla_passed,
        "sla_failed":   len(sla_results) - sla_passed,
    },
    "rules": results,
    "sla":   sla_results,
}

ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = Path(f"output/reports/report_{ts}.json")
report_path.write_text(json.dumps(report, indent=2, default=str))

print(f"Report saved: {report_path}")
print()
for k, v in report["summary"].items():
    print(f"  {k:<18} {v}")

In [ ]:
# Print alert
rule_failures = [r for r in results     if not r["passed"]]
sla_failures  = [s for s in sla_results if not s["passed"]]

if not rule_failures and not sla_failures:
    print("✓ All checks passed — no alert needed.")
else:
    print(f"✗ {len(rule_failures)} rule failures  +  {len(sla_failures)} SLA breaches\n")

    if rule_failures:
        print("RULE FAILURES:")
        for r in rule_failures:
            print(f"  [{r['table']}]  {r['rule']}  →  {r['detail']}")

    if sla_failures:
        print("\nSLA BREACHES:")
        for s in sla_failures:
            print(f"  [{s['table']}]  {s['check']}  →  {s['detail']}")

In [ ]:
# To send email alerts, fill in your SMTP details and uncomment:
#
# import smtplib
# from email.mime.text import MIMEText
#
# def send_alert(rule_failures, sla_failures):
#     body  = "DATA QUALITY ALERT\n\nRule failures:\n"
#     body += "\n".join(f"  [{r['table']}] {r['rule']} — {r['detail']}" for r in rule_failures)
#     body += "\n\nSLA breaches:\n"
#     body += "\n".join(f"  [{s['table']}] {s['check']} — {s['detail']}" for s in sla_failures)
#
#     msg            = MIMEText(body)
#     msg["Subject"] = "DQ Alert — failures detected"
#     msg["From"]    = "alerts@yourcompany.com"
#     msg["To"]      = "team@yourcompany.com"
#
#     with smtplib.SMTP("smtp.gmail.com", 587) as s:
#         s.starttls()
#         s.login("your@gmail.com", "your_app_password")
#         s.sendmail(msg["From"], [msg["To"]], msg.as_string())
#
# if rule_failures or sla_failures:
#     send_alert(rule_failures, sla_failures)
print("Email block is ready — just uncomment and add your SMTP credentials.")

## Quick Reference

```python
# Validation rules
check_not_null(df, "col")
check_unique(df, "col")
check_range(df, "col", min_val=0, max_val=100)
check_format(df, "col", r"regex", "label")
check_values(df, "col", ["a", "b", "c"])
check_fk(df, "col", parent_df, "parent_col")
check_no_future(df, "date_col")
check_no_duplicates(df)
check_row_count(df, min_rows=1, max_rows=1_000_000)

# SLA config keys
SLA = {
    "my_table": {
        "max_age_hours":   24,
        "min_pass_rate":   90,
        "min_rows":        1,
        "max_rows":        1_000_000,
        "max_runtime_sec": 300,
    }
}
```

**To use on your own data**, replace the sample DataFrames at the top with:
```python
df = pd.read_csv("your_file.csv")
# or
df = pd.read_sql("SELECT * FROM your_table", conn)
```
Then adjust the allowed values and ranges to match your business rules.